# 41. 실전 Segmentation 프로젝트 개요

6장은 SegFormer 구조 이해에서 한 단계 더 나아가, 실제 image-mask 데이터셋으로 semantic segmentation 모델을 학습하고 평가하는 파이프라인을 다룹니다.

이번 노트북의 목표는 다음과 같습니다.

- 실전 segmentation 프로젝트의 전체 흐름을 정리합니다.
- 데이터, 모델, loss, metric, 시각화가 어디에 연결되는지 파악합니다.
- 이후 노트북에서 구현할 구성 요소를 미리 배치합니다.

## 41-1. 6장의 위치

지금까지는 모델이 어떤 구조를 갖는지 이해하는 데 집중했습니다.

```text
1장: classification
2장: detection
3장: segmentation
4장: Transformer vision model
5장: SegFormer structure
6장: 실제 segmentation 학습 파이프라인
```

6장의 핵심 질문은 다음입니다.

```text
SegFormer 같은 segmentation 모델을 실제 데이터셋에 어떻게 학습시키고,
어떤 metric으로 평가하며,
어떤 실패 사례를 보고 개선할 것인가?
```

In [ ]:
pipeline = [
    "project goal",
    "dataset structure",
    "Dataset / DataLoader",
    "model forward",
    "loss",
    "train / validation loop",
    "metric",
    "visualization",
    "error analysis",
]

for i, step in enumerate(pipeline, start=1):
    print(f"{i}. {step}")

## 41-2. Segmentation 프로젝트의 입력과 출력

Semantic segmentation 학습의 기본 단위는 image와 mask입니다.

| 항목 | 예시 shape | 의미 |
|---|---:|---|
| image | `3, H, W` | RGB 입력 이미지 |
| mask | `H, W` | 각 픽셀의 class id |
| logits | `C, H, W` | 모델이 예측한 class별 점수 |
| prediction | `H, W` | `argmax(logits)` 결과 |

mask는 일반 이미지처럼 보일 수 있지만, 실제로는 색이 아니라 class id를 담은 label map입니다.

In [ ]:
import numpy as np

height, width, num_classes = 128, 128, 4
image = np.random.rand(3, height, width).astype("float32")
mask = np.random.randint(0, num_classes, size=(height, width), dtype="int64")
logits = np.random.randn(num_classes, height, width).astype("float32")
prediction = logits.argmax(axis=0)

print("image:", image.shape, image.dtype)
print("mask:", mask.shape, mask.dtype)
print("logits:", logits.shape, logits.dtype)
print("prediction:", prediction.shape, prediction.dtype)

## 41-3. 실전에서 자주 깨지는 지점

실전 segmentation 파이프라인은 모델 구조보다 데이터 처리에서 더 자주 문제가 생깁니다.

- image resize와 mask resize 방식을 다르게 처리해야 합니다.
- mask에는 RGB 색상이 아니라 class id가 들어가야 합니다.
- loss와 metric에서 `ignore_index` 처리가 일치해야 합니다.
- train/validation split이 이미지와 mask 쌍을 깨뜨리면 안 됩니다.
- pixel accuracy만 보면 class imbalance 문제를 놓칠 수 있습니다.

## 정리

- 6장은 SegFormer를 실제 프로젝트로 옮기는 단계입니다.
- 핵심 흐름은 dataset, dataloader, train loop, validation, metric, visualization입니다.
- 다음 노트북 `42_데이터셋_구조와_Annotation_이해.ipynb`에서는 image-mask 데이터셋의 폴더 구조와 annotation 형식을 다룹니다.